In [4]:
"""Drop artifact/empty columns, check for label leakage via missingness, re-run baseline models.

Usage:  python rerun_clean_models.py
Edit PATHS and SAMPLE_NET below. Raw CSVs are never modified; outputs go to OUT_DIR.
"""
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

SEED = 42
BASE = Path(r"D:\Dharini\Capstone")

PATHS = {
    "host": BASE / "Host Events" / "EVSE-B-HPC-Kernel-Events-Combined.csv",
    "netA": BASE / "Network Traffic" / "EVSE-A-Network-Combined.csv",
    "netB": BASE / "Network Traffic" / "EVSE-B-Network-Combined.csv",
    "power": BASE / "Power Consumption" / "EVSE-B-PowerCombined.csv",
}
OUT_DIR = Path("clean_rerun_results")
OUT_DIR.mkdir(exist_ok=True)
SAMPLE_NET = 60_000          # rows per network file, drawn with a cap per attack type
MIN_CLASS_ROWS = 200         # classes smaller than this are not modelled
NET_DROP = ["server_fingerprint", "client_fingerprint", "content_type",
            "user_agent", "requested_server_name"]

RECON = {"port-scan", "service-detection", "os-fingerprinting", "vuln-scan",
         "aggressive-scan", "syn-stealth"}
DOS = {"tcp-flood", "udp-flood", "syn-flood", "icmp-flood", "icmp-fragmentation",
       "push-ack-flood", "synonymous-ip-flood", "slowloris"}
HOST_ATK = {"backdoor", "cryptojacking"}


def find_col(df, *names):
    low = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in low:
            return low[n.lower()]
    raise KeyError(f"none of {names} in columns: {list(df.columns)[:40]} ...")


# Label repairs seen in the raw HPC file (confirm with the printout in the HOST section).
ATTACK_FIX = {
    "0": "backdoor",                      # backdoor rows carry 0 in the Attack column
    "serice-detection": "service-detection",
    "os-scan": "os-fingerprinting",
    "icmp-fragmentation-old": "icmp-fragmentation",
}
META_COLS = {"attack", "label", "scenario", "state", "interface", "time", "timestamp", "date"}


def norm_attack(s):
    s = s.astype(str).str.strip().str.lower().str.replace(r"[\s_]+", "-", regex=True)
    return s.replace(ATTACK_FIX)


def coerce_numeric(df):
    """The raw HPC CSV has stray text rows, so every column loads as text. Convert the feature columns."""
    for c in df.columns:
        if c.lower() not in META_COLS and c != "attack_norm" and c != "group":
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def to_group(a):
    if a in RECON: return "recon"
    if a in DOS: return "DoS"
    if a in HOST_ATK: return "host-attack"
    if a in {"benign", "none", "nan"}: return "none"
    return "other"


def log_drop(name, cols):
    print(f"[{name}] dropping {len(cols)} columns: {cols[:8]}{' ...' if len(cols) > 8 else ''}")
    pd.Series(cols, name="dropped").to_csv(OUT_DIR / f"{name}_dropped_columns.csv", index=False)


def blocked_split_per_class(y, train_frac=0.7):
    """Within each class keep file order: first 70% of its rows train, last 30% test.
    Neighbouring (near-duplicate) rows stay on the same side, and every class appears in both sets.
    A Scenario-grouped split failed here because some classes have a single scenario."""
    y = pd.Series(y).reset_index(drop=True)
    tr, te = [], []
    for _, idx in y.groupby(y).groups.items():
        idx = np.sort(np.asarray(idx))
        cut = max(1, int(len(idx) * train_frac))
        tr.extend(idx[:cut]); te.extend(idx[cut:])
    return np.array(sorted(tr)), np.array(sorted(te))


def fit_eval(name, task, X, y, groups=None):
    X = X.replace([np.inf, -np.inf], np.nan)
    tr, te = blocked_split_per_class(y)
    models = {
        "Dummy": DummyClassifier(strategy="most_frequent"),
        "LogReg": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                LogisticRegression(max_iter=2000)),
        "RandomForest": make_pipeline(SimpleImputer(strategy="median"),
                                      RandomForestClassifier(200, n_jobs=-1, random_state=SEED)),
    }
    rows = []
    for mname, m in models.items():
        t0 = time.time()
        m.fit(X.iloc[tr], y.iloc[tr])
        p = m.predict(X.iloc[te])
        rows.append(dict(experiment=f"{name}_{task}", model=mname,
                         n_train=len(tr), n_test=len(te),
                         accuracy=round(accuracy_score(y.iloc[te], p), 4),
                         macro_F1=round(f1_score(y.iloc[te], p, average="macro"), 4),
                         train_s=round(time.time() - t0, 2)))
    return rows


def numeric_features(df, exclude):
    X = df.drop(columns=[c for c in exclude if c in df.columns]).select_dtypes("number")
    keep = [c for c in X.columns if X[c].nunique(dropna=True) > 1]
    return X[keep]


def sample_by_class(df, col, total):
    per = max(total // max(df[col].nunique(), 1), 1)
    parts = [g.sample(min(len(g), per), random_state=SEED) for _, g in df.groupby(col)]
    return pd.concat(parts).sort_index()   # keep original file order for the blocked split


results = []

# ---------- HOST ----------
host = pd.read_csv(PATHS["host"], low_memory=False)
unnamed = [c for c in host.columns if str(c).startswith("Unnamed")]
host = host.drop(columns=unnamed)
log_drop("host", unnamed)
atk_col = find_col(host, "attack", "Attack")
lab_col = find_col(host, "label", "Label")
host = host.dropna(subset=[atk_col]).copy()          # stray blank rows
stray = host[atk_col].astype(str).str.contains(":", regex=False)   # e.g. 'writeback:writeback-write-inode'
print(f"[host] dropping {int(stray.sum())} stray rows with ':' in the Attack column")
host = host[~stray].copy()
print("\n[host] Label values for rows whose Attack column is 0 (confirm these are backdoor):")
print(host.loc[host[atk_col].astype(str).str.strip() == "0", lab_col].value_counts())
host = coerce_numeric(host).copy()
host["attack_norm"] = norm_attack(host[atk_col])
host["group"] = host["attack_norm"].map(to_group)
scen_col = next((c for c in host.columns if c.lower() == "scenario"), None)

# leakage check: is missingness aligned with the label?
for c in ["writeback_flush_foreign"]:
    if c in host.columns:
        print(f"\n[host] missingness of {c} by attack_norm (leak check):")
        print(host.groupby(host[c].isna())["attack_norm"].value_counts().unstack(0).fillna(0).astype(int).head(20))

host_no_bd = host[host["attack_norm"] != "backdoor"]
excl = [atk_col, lab_col, "attack_norm", "group"] + ([scen_col] if scen_col else [])
for tag, d in [("host", host), ("host_no_backdoor", host_no_bd)]:
    X = numeric_features(d, excl)
    g = d[scen_col] if scen_col else None
    print(f"[{tag}] features: {X.shape[1]}")
    results += fit_eval(tag, "binary", X, (d["group"] != "none").map({True: "attack", False: "benign"}), g)
    results += fit_eval(tag, "group", X, d["group"], g)

# ---------- POWER ----------
power = pd.read_csv(PATHS["power"], low_memory=False)
p_atk = find_col(power, "attack", "Attack")
power["attack_norm"] = norm_attack(power[p_atk])
power["group"] = power["attack_norm"].map(to_group)
power["is_attack"] = np.where(power["group"] == "none", "benign", "attack")
num = power.select_dtypes("number").columns.tolist()
state = next((c for c in power.columns if c.lower() == "state"), None)
Xp = power[num].copy()
if state:
    Xp["state_charging"] = (power[state].astype(str).str.lower() == "charging").astype(int)
for c in [c for c in num if c.lower() in {"current_ma", "power_mw"}]:   # rolling windows
    for w in (10, 30):
        Xp[f"{c}_mean{w}"] = power[c].rolling(w, min_periods=1).mean()
        Xp[f"{c}_std{w}"] = power[c].rolling(w, min_periods=1).std().fillna(0)
# The power file is ordered by scenario, so a single 70/30 cut of the whole file gives a test set with
# only one or two classes (a first run scored the majority baseline 1.0000). Split inside each attack
# type instead: first 70% of its rows train, last 30% test.
ptr, pte = blocked_split_per_class(power["attack_norm"])
for task, y in [("binary", power["is_attack"]), ("group", power["group"])]:
    tr, te = ptr, pte
    for mname, m in {"Dummy": DummyClassifier(strategy="most_frequent"),
                     "RandomForest": RandomForestClassifier(200, n_jobs=-1, random_state=SEED)}.items():
        m.fit(Xp.iloc[tr], y.iloc[tr]); p = m.predict(Xp.iloc[te])
        results.append(dict(experiment=f"power_{task}_windowed_blocked", model=mname,
                            n_train=len(tr), n_test=len(te),
                            accuracy=round(accuracy_score(y.iloc[te], p), 4),
                            macro_F1=round(f1_score(y.iloc[te], p, average="macro"), 4)))

# ---------- NETWORK ----------
for key in ("netA", "netB"):
    df = pd.read_csv(PATHS[key], low_memory=False)
    drop = [c for c in NET_DROP if c in df.columns]
    drop += [c for c in df.columns if c not in drop and df[c].nunique(dropna=False) <= 1]
    df = df.drop(columns=drop)
    log_drop(key, drop)
    a = find_col(df, "attack", "Attack", "attack_type")
    df["attack_norm"] = norm_attack(df[a])
    df["group"] = df["attack_norm"].map(to_group)
    counts = df["attack_norm"].value_counts()
    keep = counts[counts >= MIN_CLASS_ROWS].index
    df = df[df["attack_norm"].isin(keep)]
    df = sample_by_class(df, "attack_norm", SAMPLE_NET)
    # drop port columns and obvious identifiers to avoid shortcuts
    # ports, identifiers and absolute timestamps (NFStream *_seen_ms) identify the capture, not the behaviour
    leak = [c for c in df.columns if "port" in c.lower() or "seen_ms" in c.lower()
            or c.lower() in {"id", "expiration_id", "src_ip", "dst_ip", "src_mac", "dst_mac", "timestamp"}]
    print(f"[{key}] excluding leakage-prone columns: {leak}")
    excl = [a, "attack_norm", "group"] + [c for c in df.columns if "label" in c.lower()] + leak
    X = numeric_features(df, excl)
    print(f"[{key}] features: {X.shape[1]}, rows: {len(df)}")
    results += fit_eval(f"{key}_noport", "group", X, df["group"])
    results += fit_eval(f"{key}_noport", "attack_type", X, df["attack_norm"])
    del df

out = pd.DataFrame(results)
out.to_csv(OUT_DIR / "metrics_clean_rerun.csv", index=False)
print("\n", out.to_string(index=False))
print(f"\nSaved to {OUT_DIR.resolve()}")

[host] dropping 4 columns: ['Unnamed: 911', 'Unnamed: 912', 'Unnamed: 913', 'Unnamed: 914']
[host] dropping 1 stray rows with ':' in the Attack column

[host] Label values for rows whose Attack column is 0 (confirm these are backdoor):
Label
0    2302
Name: count, dtype: int64

[host] missingness of writeback_flush_foreign by attack_norm (leak check):
writeback_flush_foreign  False  True 
attack_norm                          
aggressive-scan            182      0
backdoor                     0   2302
cryptojacking             1793      0
icmp-flood                 128      0
icmp-fragmentation         180      0
none                      2302      0
os-fingerprinting          182      0
port-scan                  201      0
push-ack-flood             127      0
service-detection          199      0
syn-flood                  126      0
syn-stealth                189      0
synonymous-ip-flood        122      0
tcp-flood                  123      0
udp-flood                  120      0


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Dharini\\Capstone\\Network Traffic\\EVSE-A-Network-Combined.csv'